<a href="https://colab.research.google.com/github/mbc2009/MKsim_HL/blob/codex%2Fview-repository-contents/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from tqdm import tqdm


# New Section

In [22]:
import numpy as np
from dataclasses import dataclass, field

@dataclass
class data_block():
  row: int
  column: int
  b:int     = 2     # 单元素字节数 (单位: Byte) 默认 fp16 => b=2
  prec:int  = 16    # 数据精度
  @property
  def size(self):
    """计算数据总字节数"""
    return self.row * self.column * self.b # (单位: byte)

@dataclass
class item_emb_class(data_block):
  K_emb: float       = 2e3      # 候选物品数
  dim_item: int    = 1024     # 物品维度
  row: int         = field(default_factory=lambda: int(item_emb_class.K_emb))
  column: int      = field(default_factory=lambda: int(item_emb_class.dim_item))

@dataclass
class user_emb_class(data_block):
  S: int           = int(8e3)    # 用户序列长
  dim_item: int    = 1024        # 物品维度
  dim_user: int    = 512         # 用户行为维度，这里修改为带类型提示的字段
  column: int      = field(default_factory=lambda: int(user_emb_class.dim_user + user_emb_class.dim_item))
  row: int         = field(default_factory=lambda: int(user_emb_class.S))




# 打印
item_emb = item_emb_class()
print(f"item_emb.row: {item_emb.row}")
print(f"item_emb.column: {item_emb.column}")
print(f"item_emb.size: {item_emb.size} Bytes\n")

user_emb = user_emb_class()
print(f"user_emb.row: {user_emb.row}")
print(f"user_emb.column: {user_emb.column}")
print(f"user_emb.size: {user_emb.size} Bytes")

item_emb.row: 2000
item_emb.column: 1024
item_emb.size: 4096000 Bytes

user_emb.row: 8000
user_emb.column: 1536
user_emb.size: 24576000 Bytes


In [23]:
def transfer_latency(data_trans,T_0=200):
  data_trans = data_trans # Byte
  flit = 64   # Byte
  T_0  = T_0  # ns
  BW_eff = 32 # GB/s
  T_total = T_0/1e6 + (int(data_trans/flit) * flit)/ (BW_eff * 1e9) * 1e3
  return T_total # ms

def T_end2end(num_trans,T_trans):
  return num_trans * T_trans # ms

In [32]:
# user emb transfer time latency
user_emb = user_emb_class()

for t_0 in  np.arange(100,500,50):
  # part1: user action part
  num_fetch  = user_emb.row
  size_fetch = user_emb.size/user_emb.row * (user_emb.dim_user/(user_emb.dim_user + user_emb.dim_item))
  T_user_emb_1 = T_end2end(
      num_trans=num_fetch,
      T_trans=transfer_latency(size_fetch,t_0)
  )
  # part2: item tensor
  num_fetch  = user_emb.row
  size_fetch = user_emb.size/user_emb.row * (user_emb.dim_item/(user_emb.dim_user + user_emb.dim_item))
  T_user_emb_2 = T_end2end(
      num_trans=num_fetch,
      T_trans=transfer_latency(size_fetch,t_0)
  )

  # sun
  T_user_emb = T_user_emb_1 + T_user_emb_2

  print(f'T_0={t_0:.0f} ns  T={T_user_emb:.2f} ms')

T_0=100 ns  T=2.37 ms
T_0=150 ns  T=3.17 ms
T_0=200 ns  T=3.97 ms
T_0=250 ns  T=4.77 ms
T_0=300 ns  T=5.57 ms
T_0=350 ns  T=6.37 ms
T_0=400 ns  T=7.17 ms
T_0=450 ns  T=7.97 ms


In [28]:
item_emb = item_emb_class()
for t_0 in  np.arange(100,500,50):
  num_fetch  = item_emb.row
  size_fetch = item_emb.size/item_emb.row
  T_item_emb = T_end2end(
      num_trans=num_fetch,
      T_trans=transfer_latency(size_fetch,t_0)
  )
  print(f'T_0={t_0:.0f}  T={T_item_emb:.2f} ms')

T_0=100  T=0.33 ms
T_0=150  T=0.43 ms
T_0=200  T=0.53 ms
T_0=250  T=0.63 ms
T_0=300  T=0.73 ms
T_0=350  T=0.83 ms
T_0=400  T=0.93 ms
T_0=450  T=1.03 ms
